In [1]:
#mount google drive
from google.colab import drive
import os
drive.mount('/content/drive')
DATA_PATH  = os.path.join(os.getcwd(),'drive', 'MyDrive', 'DSE_260A', 'data')

Mounted at /content/drive


In [2]:
# Import required libraries
import os
import pandas as pd

In [4]:
# Load the dataset and select relevant columns
df = pd.read_csv(os.path.join(DATA_PATH,'challenge_participants.tsv'), sep='\t')
df = df[['participant_id', 'biological_sex', 'race', 'min_age', 'max_age', 'geolocation', 'arm_name']]
df.head()

,participant_id,biological_sex,race,min_age,max_age,geolocation,arm_name
0,2024_UGA.ID_077,female,race: unknown,72,72,US: Georgia,2024 UGA High Dose Fluzone
1,2024_UGA.ID_086,male,race: unknown,65,65,US: Georgia,2024 UGA High Dose Fluzone
2,2024_UGA.ID_128,female,race: unknown,43,43,US: Georgia,2024 UGA Standard Fluzone
3,2024_UGA.ID_170,female,race: unknown,74,74,US: Georgia,2024 UGA High Dose Fluzone
4,2024_UGA.ID_179,female,race: unknown,70,70,US: Georgia,2024 UGA High Dose Fluzone


In [5]:
# Feature engineering: Calculate average age and drop original min/max age columns
df['age'] = (df['min_age'] + df['max_age']) / 2
df = df.drop(columns=['min_age', 'max_age'])

# Clean biological_sex column (Standardize Male/Female)
df['biological_sex'] = df['biological_sex'].str.capitalize()

# Clean race column (Group races into specific categories)
race_map = {
    'Black or African American': 'Black',
    'American Indian or Alaska Native': 'American Indian',
    'race: other': 'Other',
    'race: unknown': 'Unknown',
    'Native Hawaiian or Other Pacific Islander': 'Pacific Islander'
}
df['race'] = df['race'].replace(race_map)

# Clean geolocation column (Extract state name only)
df['geolocation'] = df['geolocation'].replace({'US: US: Maryland/Virginia/DC': 'Virginia'})
df['geolocation'] = df['geolocation'].str.replace('US: ', '', regex=False)

# Simplify arm_name to vaccine dose category
def simplify_arm(arm):
    if isinstance(arm, str):
        if 'High Dose' in arm:
            return 'High Dose Fluzone'
        elif 'Standard' in arm or 'standard' in arm:
            return 'Standard Fluzone'
    return 'Other'

df['arm_name'] = df['arm_name'].apply(simplify_arm)

df.head()

,participant_id,biological_sex,race,geolocation,arm_name,age
0,2024_UGA.ID_077,Female,Unknown,Georgia,High Dose Fluzone,72.0
1,2024_UGA.ID_086,Male,Unknown,Georgia,High Dose Fluzone,65.0
2,2024_UGA.ID_128,Female,Unknown,Georgia,Standard Fluzone,43.0
3,2024_UGA.ID_170,Female,Unknown,Georgia,High Dose Fluzone,74.0
4,2024_UGA.ID_179,Female,Unknown,Georgia,High Dose Fluzone,70.0


In [6]:
# Print unique values for each column to verify cleaning
for col in df.columns:
    print(f"{col}: {df[col].unique()}\n")

participant_id: ['2024_UGA.ID_077' '2024_UGA.ID_086' '2024_UGA.ID_128' '2024_UGA.ID_170'
 '2024_UGA.ID_179' '2024_UGA.ID_215' '2024_UGA.ID_219' '2024_UGA.ID_275'
 '2024_UGA.ID_295' '2024_UGA.ID_296' '2024_UGA.ID_306' '2024_UGA.ID_313'
 '2024_UGA.ID_321' '2024_UGA.ID_335' '2024_UGA.ID_340' '2024_UGA.ID_352'
 '2024_UGA.ID_353' '2024_UGA.ID_368' '2024_UGA.ID_404' '2024_UGA.ID_411'
 '2024_UGA.ID_412' '2024_UGA.ID_430' '2024_UGA.ID_432' '2024_UGA.ID_438'
 '2024_UGA.ID_443' '2024_UGA.ID_448' '2024_UGA.ID_455' '2024_UGA.ID_461'
 '2024_UGA.ID_507' '2024_UGA.ID_512' '2024_UGA.ID_572' '2024_UGA.ID_586'
 '2024_UGA.ID_589' '2024_UGA.ID_594' '2024_UGA.ID_600' '2024_UGA.ID_614'
 '2024_UGA.ID_629' '2024_UGA.ID_632' '2024_UGA.ID_649' '2024_UGA.ID_683']

biological_sex: ['Female' 'Male']

race: ['Unknown']

geolocation: ['Georgia']

arm_name: ['High Dose Fluzone' 'Standard Fluzone']

age: [72. 65. 43. 74. 70. 45. 80. 71. 66. 55. 63. 77. 76. 53. 24. 60. 42. 35.
 54. 89. 44. 27. 31. 67. 62. 87. 19.]



In [7]:
df.columns = ['participant_id'] + [f'PART_{c}' for c in df.columns[1:]]
df.head()

,participant_id,PART_biological_sex,PART_race,PART_geolocation,PART_arm_name,PART_age
0,2024_UGA.ID_077,Female,Unknown,Georgia,High Dose Fluzone,72.0
1,2024_UGA.ID_086,Male,Unknown,Georgia,High Dose Fluzone,65.0
2,2024_UGA.ID_128,Female,Unknown,Georgia,Standard Fluzone,43.0
3,2024_UGA.ID_170,Female,Unknown,Georgia,High Dose Fluzone,74.0
4,2024_UGA.ID_179,Female,Unknown,Georgia,High Dose Fluzone,70.0


In [9]:
# Save the cleaned dataset to the cleaned_data folder
os.makedirs('../cleaned_data', exist_ok=True)
df.to_csv(os.path.join(DATA_PATH, 'cleaned_data', 'challenge_participants_cleaned.csv'), index=False)